In [0]:
import json
from pyspark.sql.functions import udf,col
from pyspark.sql.types import StringType

df_datadictionary = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_datadictionary")

# Parse datafield_datasourcePath (JSON array like ["22","2222","22"]) into path string \22\2222\22
def parse_path(raw):
    if not raw:
        return None
    try:
        parsed = json.loads(raw)
        # Handle {"value": ["folder1", "folder2"]} format
        if isinstance(parsed, dict) and 'value' in parsed:
            items = parsed['value']
        elif isinstance(parsed, list):
            items = parsed
        else:
            return None
        if items and isinstance(items, list):
            path = '\\' + '\\'.join(items)
            return path
        return None
    except Exception:
        return None

parse_path_udf = udf(parse_path, StringType())


df_datadictionary = df_datadictionary.withColumn("datafield_datasourcePath", parse_path_udf(col("datafield_datasourcePath")))

# Dedup: keep only the latest ingestion_ts per datafield to avoid row inflation downstream
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

dedup_window = Window.partitionBy("Document_Id", "Data_Provider_ID", "datafield_id").orderBy(col("ingestion_ts").desc())
df_datadictionary = df_datadictionary.withColumn("_rank", row_number().over(dedup_window)).filter(col("_rank") == 1).drop("_rank")


df_datadictionary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_datadictionary_temp")

display(df_datadictionary)

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage as(
    select wdd1.*, clean_cms.Universe_ID, clean_cms.Universe_Name, clean_cms.webi_all_sql_queries from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_datadictionary_temp as wdd1 left join
    (select Document_Id, Data_Provider_ID, DataSource_Type, DataSource_ID as Universe_ID, DataSource_Name as Universe_Name, string_agg(SQL_Query, ' | ') as webi_all_sql_queries
    from (
        select *
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
        qualify row_number() over (partition by Document_Id, Data_Provider_ID, SQL_Index order by ingestion_ts desc) = 1
    )
    group by Document_Id, Data_Provider_ID, DataSource_Type, DataSource_ID, DataSource_Name) as clean_cms
    on trim(wdd1.Document_Id) = trim(clean_cms.Document_Id) and trim(wdd1.Data_Provider_ID) = trim(clean_cms.Data_Provider_ID)
);
select 'before' as ca, count(*) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_datadictionary_temp
union all
select 'after' as ca, count(*) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage


In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage as(

    select 
        va2.Document_Id,
        va2.Document_name,
        va2.Data_Provider_ID,
        va2.datafield_id,
        va2.datafield_name,
        va2.datafield_qualification,
        va2.datafield_datatype,
        wd2.DataItem_name as datafield_name_universe,
        va2.datafield_datasourceObjectId,
        wd2.DataItem_id,
        va2.DataSource_Name,
        wd2.universe_name,
        wd2.DataItem_description as datafield_description,
        va2.datafield_datasourcepath,
        wd2.DataItem_path,
        wd2.sql_definition
    from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage as va2
    left join (
        select
            universe_name,
            DataItem_id,
            DataItem_name,
            DataItem_description,
            DataItem_path,
            DataItem_type,
            DataItem_dataType,
            universe_type,
            sql_definition
        from (select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition
        qualify row_number() over(partition by universe_name, DataItem_id order by universe_id desc) = 1)
    ) as wd2
        on lower(
            case 
                when lower(va2.DataSource_Name) like '%_old_udt' 
                then regexp_replace(va2.DataSource_Name, '_old_udt$', '') 
                else va2.DataSource_Name 
            end
        ) = lower(wd2.universe_name)
        and (
            va2.datafield_datasourceObjectId = wd2.DataItem_id 
            or (
                va2.datafield_name = wd2.DataItem_name 
                and va2.datafield_datasourcepath = wd2.DataItem_path
            )
        )
    qualify row_number() over (
        partition by va2.Document_Id, va2.Data_Provider_ID, va2.datafield_id, va2.datafield_name, va2.datafield_datasourcepath
        order by CASE WHEN va2.datafield_datasourceObjectId = wd2.DataItem_id THEN 0 ELSE 1 END
    ) = 1
)
;
select 'before' as ca, count(*) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_datadictionary_temp
union all
select 'after' as ca, count(*) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_dictionary_linage


**BELOW Cells used to prepare WEBI level tables for PowerBI**

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_audit_history as (
    with audit_history as (
        select
            WEBI_Doc_ID,
            event_type,
            upper(coalesce(UserName, 'Non_User')) as UserName,
            Access_TS,
            sum(Events) as interactions
        from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit
        group by
            WEBI_Doc_ID,
            event_type,
            coalesce(UserName, 'Non_User'),
            Access_TS
    )
    select
        WEBI_Doc_ID,
        event_type,
        upper(trim(concat(trim(WEBI_Doc_ID), '_', trim(UserName)))) as composite_ID,
        case
            when event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
            else 'Viewer_events'
        end as usage_category,
        UserName,
        Access_TS,
        interactions
    from audit_history
)


In [0]:
%sql
-- ALTER TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_audit_history ADD COLUMN IF NOT EXISTS Report_User_Mode STRING;
-- describe table dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_audit_history
update dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_audit_history
Set Report_User_Mode = CASE
    WHEN EXISTS (
        SELECT 1
        FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_audit_history as ah2
        WHERE ah2.WEBI_Doc_ID = PBI_WEBI_audit_history.WEBI_Doc_ID
          AND ah2.UserName = PBI_WEBI_audit_history.UserName
          AND ah2.usage_category = 'Editor_events'
    )
    THEN 'Edited'
    ELSE 'Viewed'
END;

select  composite_ID, count(distinct report_user_mode) from _sqldf
group by composite_ID
having  count(distinct report_user_mode) > 1

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_Basic as
(
    with Pre_full_list as (
    -- Owner Document IDs
    select distinct
        cms1.Document_Id,
        UPPER(coalesce(ud1.user_ID, 'Owner_left')) as user_id
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
        on upper(trim(ud1.user_ID)) = upper(trim(
        case 
            when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
            else cms1.created
        end
        ))
    where cms1.Document_name is not null
        and cms1.Document_name != 'Document Not Found on Server'
        -- and ud1.user_ID is not null

    union all

    -- Viewing Document IDs
    select distinct
        cms1.Document_Id,
        UPPER(coalesce(uaa1.UserName, 'Non_User')) as user_id
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
    left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
        on trim(uaa1.WEBI_Doc_ID) = trim(cms1.Document_Id)
    -- left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
    --     on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
    where cms1.Document_name is not null
        and cms1.Document_name != 'Document Not Found on Server'
        -- and ud1.user_ID is not null
    ),
    full_list as (
    select
        distinct
        document_id,
        user_id,
        upper(trim(concat(cast(document_id as string), '_', trim(user_id)))) as composite_ID
    from Pre_full_list
    )
    select
    fl1.document_id,
    fl1.user_id,
    fl1.composite_ID,
    cms2.Document_name,
    cms2.Full_path as WEBI_folder_path,
    cms2.scheduled as Had_schedule,
    cms2.Number_of_Data_Providers,
    se1.Active_Schedule as schedule_active,
    aa2.interactions as Audit_interactions,
    case when fl1.composite_ID in (select distinct composite_ID from dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_audit_history where composite_ID is not null) then True else false end as Composite_Flag
    from full_list as fl1
    left join (select * from (
    select 
        Document_Id, 
        Document_name, 
        Full_path, 
        Number_of_Data_Providers,
        scheduled,
        row_number() over (partition by Document_Id order by ingestion_ts desc, Full_path desc) as rank
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms)
    where rank = 1
    ) as cms2
    on trim(cms2.Document_Id) = trim(fl1.Document_Id)
    left join (select WEBI_Doc_ID, UserName, sum(Events) as interactions from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit where WEBI_Doc_ID is not null group by WEBI_Doc_ID, UserName )as aa2
      on trim(aa2.WEBI_Doc_ID) = trim(fl1.Document_Id) and upper(trim(aa2.UserName)) = upper(trim(fl1.user_id))
    left join (select distinct document_id, Active_Schedule from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted) as se1
      on trim(se1.document_id) = trim(fl1.Document_Id)
    -- left join  as ah1
    -- on trim(ah1.composite_ID)=trim(fl1.composite_ID)
);

select count(*) as total, count(distinct document_id , user_id) as total_combinations  
 from dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_Basic


In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.PBI_WEBI_profile as (
    with webi_basic as (
        select distinct
            Document_Id,
            Document_name,
            Full_path as LaunchPad_folder,
            scheduled,
            updated as lastUpdate_TS,
            created as Author,
            lastAuthor as lastUpdata_by,
            Number_of_Data_Providers,
            Data_Provider_ID,
            Data_Provider_Name,
            DataSource_ID,
            DataSource_Type,
            DataSource_Name
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
        qualify row_number() over (
            partition by Document_Id, Data_Provider_ID, SQL_Index
            order by ingestion_ts desc
        ) = 1
    ),
    scheduled as (
        select
            basic1.document_id,
            basic1.schedule_id,
            basic1.schedule_activeness,
            basic1.schedule_name,
            basic1.schedule_format,
            basic1.repetition_type,
            basic1.destination_type,
            scheduled1.destination_location
        from (
            select distinct
                document_id,
                schedule_id,
                schedule_name,
                schedule_format,
                repetition_type,
                destination_type,
                schedule_activeness
            from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_active
            qualify row_number() over (
                partition by document_id
                order by ingestion_ts desc
            ) = 1
        ) as basic1
        left join (
            select
                document_id,
                schedule_id,
                schedule_name,
                schedule_format,
                repetition_type,
                destination_type,
                case
                    when lower(destination_type) = 'mail' then sent_to
                    when lower(destination_type) = 'filesystem' then file_path
                    else null
                end as destination_location
            from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched
        ) as scheduled1
            on trim(basic1.document_id) = trim(scheduled1.document_id)
            and trim(basic1.schedule_id) = trim(scheduled1.schedule_id)
    )
    select webi_basic.*, scheduled.schedule_activeness, scheduled.schedule_format, scheduled.repetition_type, case when (scheduled.destination_type is null and scheduled.schedule_activeness is not null) or lower(scheduled.destination_type)='inbox' then 'SAP BO' else scheduled.destination_type end as destination_type, scheduled.destination_location
    from webi_basic
    left join scheduled
        on webi_basic.document_id = scheduled.document_id
)